[Reference](https://medium.com/area-21/modern-data-engineering-meets-mlops-building-the-backbone-of-ai-driven-products-3cecac8171be)

# Feature Engineering: The Shared Battleground

In [1]:
# feature_definitions.py — Defining features as code (versioned in Git)
from feast import Entity, Feature, FeatureView, FileSource
from feast.types import Float32, Int64
from datetime import timedelta
# The entity (primary key for feature lookup)
customer = Entity(
    name="customer_id",
    description="Unique customer identifier"
)
# The data source - points to your Silver/Gold layer
customer_transactions_source = FileSource(
    path="s3://lakehouse/gold/customer_transaction_features/",
    timestamp_field="event_timestamp"
)
# The feature view - a logical grouping of related features
customer_features = FeatureView(
    name="customer_transaction_features",
    entities=[customer],
    ttl=timedelta(days=90),  # How stale can features be?
    schema=[
        Feature(name="txn_count_30d", dtype=Int64),
        Feature(name="avg_txn_amount_30d", dtype=Float32),
        Feature(name="txn_amount_std_30d", dtype=Float32),
        Feature(name="days_since_last_txn", dtype=Int64),
    ],
    source=customer_transactions_source,
)

# The Training Pipeline: Where Orchestration Gets Serious

In [2]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import f1_score, precision_score, recall_score

# Every training run is a tracked experiment
with mlflow.start_run(run_name="fraud_model_v12") as run:
    # Log the data version - critical for reproducibility
    mlflow.log_param("data_version", "delta_version_42")
    mlflow.log_param("feature_store_snapshot", "2026-04-01")
    mlflow.log_param("training_rows", len(X_train))
    # Log hyperparameters
    params = {
        "n_estimators": 500,
        "max_depth": 8,
        "learning_rate": 0.05,
        "subsample": 0.8
    }
    mlflow.log_params(params)
    # Train
    model = GradientBoostingClassifier(**params)
    model.fit(X_train, y_train)
    # Evaluate and log metrics
    y_pred = model.predict(X_test)
    mlflow.log_metric("f1_score", f1_score(y_test, y_pred))
    mlflow.log_metric("precision", precision_score(y_test, y_pred))
    mlflow.log_metric("recall", recall_score(y_test, y_pred))
    # Log the model artifact itself
    mlflow.sklearn.log_model(
        model,
        "fraud_detector",
        registered_model_name="fraud-detection-prod"
    )

# The Monitoring Gap: Why Models Rot and What Data Engineers Can Do About It

In [3]:
from scipy.stats import ks_2samp
import pandas as pd

def detect_feature_drift(
    reference_df: pd.DataFrame,    # Training data snapshot
    production_df: pd.DataFrame,   # Recent production data
    features: list[str],
    significance_level: float = 0.05
) -> dict:
    """
    Uses the Kolmogorov-Smirnov test to detect distribution shifts
    between training data and production data for each feature.
    Returns a dict of features flagged as drifted.
    """
    drift_report = {}
    for feature in features:
        stat, p_value = ks_2samp(
            reference_df[feature].dropna(),
            production_df[feature].dropna()
        )
        drift_report[feature] = {
            "ks_statistic": round(stat, 4),
            "p_value": round(p_value, 6),
            "drifted": p_value < significance_level
        }
    drifted_features = {
        k: v for k, v in drift_report.items() if v["drifted"]
    }
    if drifted_features:
        # Trigger an alert or initiate a retraining pipeline
        print(f"⚠️  Drift detected in {len(drifted_features)} features:")
        for name, info in drifted_features.items():
            print(f"   {name}: KS={info['ks_statistic']}, p={info['p_value']}")
    return drift_report

# CI/CD for ML: It’s Not Just About Code Anymore
```
# .github/workflows/ml-ci.yml
name: ML Pipeline CI
on:
  push:
    paths:
      - 'features/**'
      - 'training/**'
      - 'models/**'
jobs:
  test-features:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Install dependencies
        run: pip install -r requirements.txt
      - name: Run feature engineering unit tests
        run: pytest tests/features/ -v
      - name: Validate feature schemas
        run: python scripts/validate_feature_schemas.py
  test-training:
    needs: test-features
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Run training pipeline on sample data
        run: python training/train.py --data-sample 1000 --epochs 2
      - name: Verify model metrics exceed thresholds
        run: python scripts/check_model_quality_gate.py
      - name: Upload model artifact
        uses: actions/upload-artifact@v4
        with:
          name: model-candidate
          path: artifacts/model/
```